# Lab 03 — Slowly Changing Dimensions Tipo 2 (DuckDB)

**Onde roda:** 🟢 Browser (JupyterLite). Execute célula a célula.

Objetivo: consultar uma dimensão **SCD2** (visão atual e *point-in-time*) e **aplicar uma mudança** encerrando a versão antiga.

In [ ]:
try:
    import duckdb
except ModuleNotFoundError:
    import piplite; await piplite.install('duckdb'); import duckdb
con = duckdb.connect()
con.execute('''
    CREATE TABLE dim_cliente(
        sk INT, cliente_id INT, cidade VARCHAR,
        valido_de DATE, valido_ate DATE, corrente BOOLEAN)
''')
con.executemany('INSERT INTO dim_cliente VALUES (?,?,?,?,?,?)', [
    (1,100,'São Paulo','2024-01-01','2025-03-01',False),
    (2,100,'Campinas','2025-03-01','9999-12-31',True),
    (3,200,'Rio de Janeiro','2024-06-01','9999-12-31',True),
    (4,300,'Curitiba','2024-02-01','2024-11-01',False),
    (5,300,'Belo Horizonte','2024-11-01','9999-12-31',True)])
con.execute('SELECT * FROM dim_cliente ORDER BY sk').df()

## 1. Visão atual (a linha corrente)
O estado de hoje: filtra `corrente = TRUE`.

In [ ]:
con.execute('''
    SELECT cliente_id, cidade
    FROM dim_cliente
    WHERE corrente
    ORDER BY cliente_id
''').df()

## 2. Point-in-time (como era numa data)
Qual a cidade de cada cliente em **01/07/2024**? Use o intervalo de vigência.

In [ ]:
con.execute('''
    SELECT cliente_id, cidade
    FROM dim_cliente
    WHERE valido_de <= DATE '2024-07-01' AND DATE '2024-07-01' < valido_ate
    ORDER BY cliente_id
''').df()

## 3. Aplicar uma mudança (o merge do SCD2)
O cliente 200 mudou para **Salvador** em 01/09/2025. Encerramos a linha corrente e inserimos a nova.

In [ ]:
# encerra a versão vigente do cliente 200
con.execute('''
    UPDATE dim_cliente
    SET valido_ate = DATE '2025-09-01', corrente = FALSE
    WHERE cliente_id = 200 AND corrente
''')
# insere a nova versão como corrente (nova surrogate key)
con.execute('''
    INSERT INTO dim_cliente VALUES
    (6, 200, 'Salvador', DATE '2025-09-01', DATE '9999-12-31', TRUE)
''')
con.execute('SELECT * FROM dim_cliente WHERE cliente_id=200 ORDER BY sk').df()

## 4. Sua vez (mini-desafio)
Depois da mudança acima, traga a **visão atual** `(cliente_id, cidade)` de todos, ordenada por `cliente_id`. Devolva `.fetchall()` e verifique.

In [ ]:
resposta = con.execute('''
    SELECT cliente_id, cidade
    FROM dim_cliente
    WHERE corrente
    ORDER BY cliente_id
''').fetchall()
resposta

In [ ]:
def verificar(rows):
    esperado = [(100,'Campinas'),(200,'Salvador'),(300,'Belo Horizonte')]
    try:
        assert rows == esperado, 'Após o merge, o cliente 200 deve estar em Salvador.'
        print('✅ Correto! Você leu a visão corrente de uma dimensão SCD2.')
    except AssertionError as e:
        print('❌', e)

verificar(resposta)